In [1]:
# ==========================================================
# ACTIVIDAD 22. ¿Por qué Gaussian Mixture agrupa distinto?
# Compara el centro y la dispersión de cada grupo en los dos
# algoritmos, y la dirección del precio dentro de cada grupo.
# Solo diagnóstico: no cambia ningún modelo.
# ==========================================================
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from google.colab import drive

VARIABLES = ["volatilidad_21", "cambio_volatilidad", "rendimiento_21", "caida_maximo_63", "volumen_anormal"]
COLUMNAS_Z = ["z_" + v for v in VARIABLES]

drive.mount("/content/drive")
RAIZ = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# ----------------------------------------------------------
# 1. Subconjunto de ajuste y los dos modelos guardados
# ----------------------------------------------------------
df = pd.read_parquet(RAIZ / "01_datos" / "variables_preparadas.parquet",
                     columns=["grupo", "bloque"] + VARIABLES + COLUMNAS_Z)
df = df[(df["grupo"] == "entrenamiento") & (df["bloque"] == "entrenamiento")].copy()
X = df[COLUMNAS_Z].to_numpy()

km = joblib.load(RAIZ / "02_modelos" / "kmeans_k3.joblib")
gm = joblib.load(RAIZ / "02_modelos" / "gmm_k3.joblib")
df["kmeans"] = pd.Series(km["modelo"].predict(X), index=df.index).map(km["renumerar"])
df["gmm"] = pd.Series(gm["modelo"].predict(X), index=df.index).map(gm["renumerar"])
# Los componentes de Gaussian Mixture se nombran por su nivel de volatilidad, no como regímenes
NOMBRES_GMM = {1: "G1 vol. baja", 2: "G2 vol. media", 3: "G3 vol. alta"}
print("Filas:", len(df))

# ----------------------------------------------------------
# 2. Centro (media) de cada grupo, en desviaciones típicas
# ----------------------------------------------------------
print("\n=== CENTRO DE CADA GRUPO (media de las variables escaladas) ===")
print("K-Means:")
print(df.groupby("kmeans")[COLUMNAS_Z].mean().rename(index=km["nombres"]))
print("Gaussian Mixture:")
print(df.groupby("gmm")[COLUMNAS_Z].mean().rename(index=NOMBRES_GMM))

# ----------------------------------------------------------
# 3. Dispersión dentro de cada grupo (desviación típica)
# ----------------------------------------------------------
print("\n=== DISPERSIÓN DENTRO DE CADA GRUPO (desviación típica de las variables escaladas) ===")
print("K-Means:")
print(df.groupby("kmeans")[COLUMNAS_Z].std().rename(index=km["nombres"]))
print("Gaussian Mixture:")
print(df.groupby("gmm")[COLUMNAS_Z].std().rename(index=NOMBRES_GMM))

# ----------------------------------------------------------
# 4. Dirección del precio dentro de cada grupo
# ----------------------------------------------------------
def direccion(grupo):
    r = grupo["rendimiento_21"]
    return pd.Series({"% mes con subida > 10 %": (r > 0.10).mean() * 100,
                      "% mes con caída > 10 %": (r < -0.10).mean() * 100,
                      "% filas": len(grupo) / len(df) * 100})

print("\n=== DIRECCIÓN DEL PRECIO EN EL ÚLTIMO MES, POR GRUPO ===")
print("K-Means:")
print(df.groupby("kmeans")[["rendimiento_21"]].apply(direccion).rename(index=km["nombres"]))
print("Gaussian Mixture:")
print(df.groupby("gmm")[["rendimiento_21"]].apply(direccion).rename(index=NOMBRES_GMM))

Mounted at /content/drive
Filas: 815829

=== CENTRO DE CADA GRUPO (media de las variables escaladas) ===
K-Means:
         z_volatilidad_21  z_cambio_volatilidad  z_rendimiento_21  z_caida_maximo_63  z_volumen_anormal
kmeans                                                                                                 
Calma               -0.42                 -0.52              0.07               0.24              -0.49
Impulso              0.05                  0.56              0.44               0.41               0.85
Estrés               1.47                  0.63             -1.29              -1.84              -0.13
Gaussian Mixture:
               z_volatilidad_21  z_cambio_volatilidad  z_rendimiento_21  z_caida_maximo_63  z_volumen_anormal
gmm                                                                                                          
G1 vol. baja              -0.64                 -0.11              0.08               0.59              -0.14
G2 vol. media     